# Phase 1 — SimCLR reference encoder, seed 1 (Colab)

Trains the Phase-1 SimCLR reference encoder for **seed 1** (the seed-to-seed reproducibility check). Checkpoints write directly to Google Drive every epoch, so a disconnected session is safe to resume.

**Before running:**
- Runtime → Change runtime type → **T4 GPU**.
- Keep the tab active or use Colab Pro/Pro+ for unattended runs — free-tier sessions disconnect after ~90 min of browser inactivity even if the kernel is busy.
- You will be prompted to authorise Google Drive access in cell 2.

In [ ]:
!nvidia-smi

## 1. Mount Google Drive

All checkpoints and final weights are written to `My Drive/probe-capacity-results/` so they persist across sessions. A browser pop-up will ask you to sign in and grant access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = "/content/drive/MyDrive/probe-capacity-results/encoders"
import os; os.makedirs(DRIVE_OUT, exist_ok=True)
print("Drive mounted. Output dir:", DRIVE_OUT)

## 2. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/content/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Runtime > Change runtime type > T4 GPU")

## 4. Download Shapes3D + build the image cache

Downloads the ~255 MB HDF5 to local SSD (`/content/`) and decompresses it once into an uncompressed memory-mapped array (~5.9 GB). Both steps are idempotent — safe to rerun.

In [ ]:
!cd /content/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 4b. Restore checkpoint from Drive (optional — only if resuming a previous run)

If a previous Colab session wrote a checkpoint to Drive, this cell copies it back to the local results directory so training resumes from where it left off. Skip if this is a fresh start.

If you're resuming from a checkpoint you downloaded from Kaggle, upload it to `My Drive/probe-capacity-results/encoders/standard_strong_seed1/last_ckpt.pt` first, then run this cell.

In [ ]:
import shutil
from pathlib import Path

RUN_ID = "standard_strong_seed1"
drive_ckpt = Path(DRIVE_OUT) / RUN_ID / "last_ckpt.pt"
local_ckpt = Path("results/encoders") / RUN_ID / "last_ckpt.pt"

if drive_ckpt.exists():
    local_ckpt.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(drive_ckpt, local_ckpt)
    print(f"restored {drive_ckpt} -> {local_ckpt}")
else:
    print(f"no checkpoint found at {drive_ckpt} — starting from epoch 0")

## 5. Train the reference encoder — seed 1

Checkpoints every epoch to local SSD (`results/encoders/standard_strong_seed1/last_ckpt.pt`) **and** to Google Drive via the sync cell below. ~10 h total on a T4. If the session disconnects, mount Drive in a fresh session, run cells 1–4 above, run the 4b restore cell, and re-run this cell — it resumes from the last saved epoch automatically.

**Do not re-run the 4b restore cell** once training has started in this session — it would overwrite the local checkpoint with the Drive copy (which may be behind if sync hasn't run yet).

In [ ]:
!cd /content/probe-capacity-invariance && python -m src.encoders.train_simclr \
    --config configs/run/reference_cuda.yaml \
    --set run.seed=1 run.num_workers=2

## 6. Sync results to Google Drive

Copies the checkpoint and log files from local SSD to Drive. Run this periodically while training is running (in a separate cell execution) to keep Drive up to date, and once more after training finishes. Safe to run multiple times.

In [ ]:
import json, shutil
from pathlib import Path

REPO_DIR = "/content/probe-capacity-invariance"
RUN_ID   = "standard_strong_seed1"
src_dir  = Path(REPO_DIR) / "results" / "encoders" / RUN_ID   # absolute — cwd-independent
dst_dir  = Path(DRIVE_OUT) / RUN_ID
dst_dir.mkdir(parents=True, exist_ok=True)

if not src_dir.exists():
    print(f"nothing to sync yet — {src_dir} not created "
          "(training hasn't finished epoch 0). Re-run after the first epoch.")
else:
    for f in sorted(src_dir.iterdir()):
        if f.is_file():
            shutil.copy2(f, dst_dir / f.name)
            print(f"synced {f.name} ({f.stat().st_size // 1024} KB)")
    # write a JSON snapshot of the latest epoch alongside the .pt on every sync
    log = src_dir / "train_log.jsonl"
    if log.exists():
        lines = [ln for ln in log.read_text().splitlines() if ln.strip()]
        if lines:
            latest = json.loads(lines[-1])
            (dst_dir / "metrics_latest.json").write_text(json.dumps(latest, indent=2))
            print(f"wrote metrics_latest.json (epoch {latest.get('epoch')})")
    print(f"done -> {dst_dir}")

## 7. Verify final output

Run after training completes. Confirms `backbone.pt` and `metrics.json` exist on Drive and prints the final metrics.

In [ ]:
import json
from pathlib import Path

RUN_ID = "standard_strong_seed1"
drive_dir = Path(DRIVE_OUT) / RUN_ID

for fname in ["backbone.pt", "metrics.json", "train_log.jsonl", "last_ckpt.pt"]:
    p = drive_dir / fname
    status = f"{p.stat().st_size // 1024} KB" if p.exists() else "MISSING"
    print(f"{fname}: {status}")

metrics_path = drive_dir / "metrics.json"
if metrics_path.exists():
    print()
    print(json.dumps(json.load(open(metrics_path)), indent=2))

## 8. Encoder-quality gate (Phase-1 exit)

Run this section **after both SimCLR seeds are trained** (seed 0 and seed 1). It:

1. **8a** — puts both SimCLR `backbone.pt` files at `results/encoders/<run_id>/backbone.pt` (from Drive if present, otherwise it prompts you to upload each one). You need **seed 0's `backbone.pt`** from your earlier run — have that file ready to upload if it isn't on Drive. Seed 1 is already on Drive from this session.
2. **8b** — trains the supervised reference encoder (seed 0), the ceiling control. Trains from scratch on the data cache — nothing to upload.
3. **8c** — runs the encoder-quality gate: random floor vs SimCLR (both seeds) vs supervised ceiling → `results/quality_gate/reference.json`, with a PASS/FAIL verdict.
4. **8d** — copies the gate report + supervised encoder to Drive and prints the verdict.

> When 8a prompts an upload, the file picker takes **one file at a time** — upload the backbone that matches the `run_id` shown in the message (both files are named `backbone.pt`, so upload them in the right order).

In [ ]:
# 8a. Ensure both SimCLR backbones are at results/encoders/<run_id>/backbone.pt.
#     Preference order per run: already local -> copy from Drive -> prompt you to upload.
import shutil
from pathlib import Path
from google.colab import files

REPO = Path("/content/probe-capacity-invariance")

def ensure_backbone(run_id):
    dst = REPO / "results" / "encoders" / run_id / "backbone.pt"
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        print(f"✓ {run_id}: already present ({dst.stat().st_size // 1024 // 1024} MB)")
        return
    drive_src = Path(DRIVE_OUT) / run_id / "backbone.pt"
    if drive_src.exists():
        shutil.copy2(drive_src, dst)
        print(f"✓ {run_id}: restored from Drive")
        return
    print(f"↑ {run_id}: not on disk or Drive — upload ITS backbone.pt now:")
    up = files.upload()                 # pick the backbone.pt for THIS run_id only
    name = next(iter(up))
    Path(name).replace(dst)             # move the uploaded file into the run dir
    print(f"✓ {run_id}: placed uploaded {name}")

for rid in ["standard_strong_seed0", "standard_strong_seed1"]:
    ensure_backbone(rid)

In [ ]:
# 8b. Train the supervised reference encoder (seed 0) — the "ceiling" control.
#     Trains from scratch on the labelled data cache (nothing to upload). Faster than SimCLR.
!cd /content/probe-capacity-invariance && python -m src.encoders.supervised \
    --config configs/run/supervised.yaml \
    --set run.seed=0 run.num_workers=2

In [ ]:
# 8c. Run the encoder-quality gate: random floor vs SimCLR (both seeds) vs supervised ceiling.
#     Prints a per-factor recoverability table, a PASS/FAIL verdict, and the
#     shape-recoverability spread across seeds, then writes results/quality_gate/reference.json.
!cd /content/probe-capacity-invariance && python -m src.eval.quality_gate \
    --config configs/run/reference_cuda.yaml \
    --simclr results/encoders/standard_strong_seed0/backbone.pt \
             results/encoders/standard_strong_seed1/backbone.pt \
    --supervised results/encoders/supervised_seed0/backbone.pt \
    --random-seed 0 \
    --out results/quality_gate/reference.json

In [ ]:
# 8d. Persist the gate report and the supervised encoder to Drive, then print the verdict.
import shutil
from pathlib import Path

REPO = Path("/content/probe-capacity-invariance")
RESULTS_ROOT = Path(DRIVE_OUT).parent          # .../probe-capacity-results

gate_src = REPO / "results" / "quality_gate" / "reference.json"
gate_dst = RESULTS_ROOT / "quality_gate" / "reference.json"
gate_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(gate_src, gate_dst)
print(f"synced {gate_dst}")

# supervised backbone (so it survives the session)
sup_src = REPO / "results" / "encoders" / "supervised_seed0" / "backbone.pt"
if sup_src.exists():
    sup_dst = Path(DRIVE_OUT) / "supervised_seed0" / "backbone.pt"
    sup_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(sup_src, sup_dst)
    print(f"synced {sup_dst}")

print("\n--- reference.json ---")
print(gate_src.read_text())